equality is adjoint to contraction
eq_{ij}(e)  the i and jth slot are the same?. Kind of a dump? Yea. Huh. It really is a dump like relation
`contract(eq_filter(e)) = contract(e)`

A different kind of contextual equality. Contextual on the equality of the incoming slots / vars


```python
def eq_filter(i,j,f):
    return lambda *args: f(*args) if args[i] == args[j] else "JUNK"

def eq(i,j,f):
    def res(*args):
        if args[i] == args[j]:
            return f(*args)
        else:
            return "JUNK"
    return res
```



In [ ]:
from enum import Enum

class Dropper(Enum):
    Drop = 0
    Keep = 1
    Prev = 2
# [Drop, Prev] is illegal
# [Keep, Prev, Prev] is legal

type CThin2 = list[int] # opposite form. Nonstrict monotonic maps. Seems kind of simple.


type CThin = list[Dropper]  # contractions and thinnings

def dom(ct: CThin) -> set[int]: return len(ct)
def cod(ct: CThin) -> set[int]: return len(1 for x in ct if x != Dropper.Drop)
def comp(t, s):
    res = []
    i = 0
    for k in t:
        if k == Dropper.Drop:
            res.append(Dropper.Drop)
        else:
            res.
            i += 1


equality adjoint to contraction
fuse_{x,y}(t) ?
fuse_{x,y}(expand_{?}(  )) = id

def fuse(i,j, f): # take in one more argument by copying
    def res(args):
        args.insert(j, args[i])
        return f(*args)
    return re
expand needs a way to distinguish uses of something. A non semantic notion?


https://arxiv.org/pdf/1807.05923 algerbaic effects. Whoa.
He has the lifting model quite explicitly


parametrized unary functions

id : R0 -> R
mul : R2 -> R

So the kind of goofball every operator is a morphism thing (lawvere) is describing an encoding that might be achievable with liftings?

Maddy asked about a constant-less version. Reserve the first 20 var / context positions. Hmm. Doesn' really work though. Any specialty equations would apply to the generic situation.


Trie Terms
Terms with 10^100 arity

Parametrized terms
Generalized arity
algebraic effects




"if you're interested in co-debruijn with bitstrings, you may be interested in the pext/pdep instructions as they work quite analogously to thinning.

pext is select bits according to a mask and then pack them gaplessly into the low bits. pdep is the inverse - take low bits and insert gaps as necessary to put them in these places.

there are analogues in masked simd too,. packing elements instead of bits. this could be used to 'perform' a thinning if you have the entries in contiguous memory."
https://social.treehouse.systems/@dysfun/116799585118532103

https://github.com/ekmett/codebruijn/blob/master/Code.hs

Thin egraph + 

Thinning Terms
A notion of terms with rigid variables


Thinning Unification
Metmath0 bound variables look a lot like thinnings. There is a bitvector of what is allow to appear or not


Nominal Unification
Alpha prolog


Binders. Do they work?

Mcbride unification paper. Thinning are evidence of termination of telescoped substitution?
Unification over thinterms. A different sense than I was intending. I was considering thinvars are rigid,  forward existentials


What the fuck.



Substructural https://en.wikipedia.org/wiki/Structural_rule

Context of dependent type theory is dependency dag?
Context in bunched logic or whatever was like a tree?
Multisets
Sets
lists




Exchange
swap_ij(t) as non baked in enode

Does this suck? Maybe

swap(f(x,y)) = f(swap(x), swap(y))



Tuple terms




The named context terms
https://www.philipzucker.com/thin_hash_cons_codebruijn/


simply typed terms


composition of thinterms.
composition by alignment on some arguments

def comp(data, f, g)

In [ ]:
from dataclasses import dataclass, field
@dataclass
class Decl:
    name : str
    intypes : list[str]
    outtypes : list[str]

@dataclass
class TApp:
    f : Decl
    args : list["TApp"]

# or inline decl
@dataclass
class TApp:
    f : str
    ty : str
    args : list["TApp"] # already contain ty


In [3]:
from dataclasses import dataclass, field
type Thin = list[bool]
def dom(f : Thin): # domain is big side
    return len(f)
def cod(f : Thin): # codomain is small side
    return sum(f)
def comp(f : Thin, g : Thin) -> Thin:
    assert cod(f) == dom(g)
    i = 0 
    result = []
    for a in f:
        if a:
            result.append(g[i])
            i += 1
        else:
            result.append(False)
    assert i == len(g)
    return result   
def lcm(f : Thin, g : Thin) -> Thin:
    assert len(f) == len(g)
    return [a and b for a,b in zip(f,g)]
def div(f : Thin, g : Thin) -> Thin:
    assert dom(f) == dom(g)
    assert all(not a for a,b in zip(f,g) if not b) # f is thinner than g
    return [a for a,b in zip(f,g) if b]

In [ ]:
from dataclasses import dataclass,replace
type Thin = list[bool]

class Term:
    pass
    def __add__(self, other):
        assert isinstance(other, Term)
        return App("+", [self, other])
    

#@dataclass
#class Var(Term):
#    lift : Thin


@dataclass
class App(Term):
    f : str
    args : list[Term]
    lift : Thin

    # __match_args__ = ("f", "children")

    def __init__(self, f: str, args: list[Term], lift: Thin = None):
        d = dom(args[0].lift) if args else 0
        assert all(dom(arg.lift) == d for arg in args)
        self.f = f
        if lift is None:
            used = [False]*d
            for arg in args:
                for i in range(d):
                    used[i] |= arg.lift[i]
            args = [replace(arg, lift=[x for keep,x in zip(used, arg.lift) if keep]) for arg in args]
            self.args = args
            self.lift = used #id(d) if lift is None else lift
        else:
            self.args = args
            self.lift = lift
    
    def children(self):
        return [replace(arg, lift=comp(self.lift, arg.lift)) for arg in self.args]
    
    def __repr__(self):
        return f"LTerm({self.f}, {self.args}, {self.lift})"

zero = App("zero", [], [])
x = App("var", [], [True, False])
y = App("var", [], [False, True])


def Vars(d : int):
    return [App("var", [], [False]*i + [True] + [False]*(d-i-1)) for i in range(d)]
def Const(name, d : int):
    return App(name, [], [False]*d)

zero = Const("zero", 3)
x,y,z = Vars(3)

x + zero + y

"""
Do I always peel off the front or can I use any var?
Do I even need to drop the var?
"""
def Lam(x : App, body : App) -> App:
    assert x.f == "var"
    t = App("lam", [x, body])
    for i in range(dom(x.lift)):
        if x.lift[i]:
            t.lift[i] = False
    return t
    #t.lift = [x for lose, x in zip(x.lift, t.lift) if not lose]

Lam(x,y)

@dataclass
class PVar(Term):
    name : str
    lift : Thin # exact match or upper bound? or lift_t(?x) so yes, upper bound 
    # d : int ?


def pmatch(t, pat) -> dict[str, Term] | None:
    subst = {}
    todo = [(t, pat)]
    while todo:
        (t, pat) = todo.pop()
        if isinstance(pat, PVar):
            # todo. Deal with lift on pvar
            if pat.name in subst:
                if subst[pat.name] != t:
                    return None
            else:
                subst[pat.name] = t
        elif isinstance(t, App) and isinstance(pat, App):
            if t.f != pat.f or len(t.args) != len(pat.args):
                return None
            todo.extend(zip(t.children(), pat.children()))
        else:
            return None
    return subst

# There is a difference if v is also in t or not.
# occurs check guarantees that n is not in t.
def subst(self, n : int, t : Term) -> Term: # return term in n-1 vars?
    assert 0 <= n < len(self.lift)
    match self:
        case App("var", [], lift):
            assert sum(lift) == 1
            if lift[n]:
                return t
            else:
                return self
        case App(f, args, lift):
            if lift[n]: # we can tell if there is something down there. I guess this is an optimization. We don't have to check it
                #n1 = sum(lift[:n]) # new index. Instead we could adjust the index and thinning on t as we go down. Is that better really?
                # change lift on t? No, we used children.
                return App(f, [subst(arg, n, t) for arg in self.children()])
            else:
                return self #App(f, args, lift[:n] + lift[n+1:])

a = PVar("a", [True, True, True])
zero + a
pmatch(zero + x, zero + a)

def unify(): ...

subst(x + zero, 0, zero + x)


LTerm(+, [LTerm(+, [LTerm(zero, [], [False]), LTerm(var, [], [True])], [True]), LTerm(zero, [], [False])], [True, False, False])

Instead of PVar also could try to use Var?



What the fuck.
Hmm. We are explicitly shadowing when we use x. That's interesting.
A different choice would be to bolt onto the front.
But also I could do that anyhow.

What is the Lam I have written semantically?
 
pattern matching inside a lambda...

Lam(z, ) de bruijn level
Lam(x, ) de bruijn index
Lam(y, ) de bruijn goofball

I am reminded of integral f du. Maybe x is kind of a "test" function?
https://en.wikipedia.org/wiki/Riemann%E2%80%93Stieltjes_integral
a meausure? volume form?

forall x : Q(x) => P(x)   quantified 
exists x, Q(x) /\ P(x)   
integral f dg
max ?
lambda 

Hmm. My variables are like director strings / maps at the binding site kind of. But I'm not mapping to posiitions in the body

so it has nothing to do with miller matching. That's still a separate mechanism...
Yes an no. By it's nature if the var captures the lambda bound one... It's just you have to be careful



lam(zero, body) ?
I'm like sampling body using x somehow to convert it into a dictionary. Some kind of curried thing?

No, but maybe the x parameter in lam is really a different thing. And we couldn't have arbitrary terms tere

x as a Pattern. lambda case.
coterm mu mu bar thing?
subterm to absrtact over? Any appearance
Lam(x : DVar, body : Term)

Interpreting a lifted lambda: 
We do have an environment list, but instead of putting variables on front and labelling with respect to front or back, we have a fixed array envinroment and we clobber it.

It's kind of interesting we can have multiple foci at once. Arrays? Vectorized?


lift_t(s) is precomposing s with R^dom -> R^cod semantic thinning function.







In [ ]:
def Lam(body):
    t = App("lam", [body])
    t.lift = t.lift[1:] # t.lift.pop()
    return t

Lam(y) # hmm. no. Need modified smart constructor
# Lam and Var should probably be different classes?


LTerm(lam, [LTerm(var, [], [True])], [True, False])

In [ ]:
def interp(t, env):
    match t:
        case App("var", [], lift):
            return thin(lift, env) # return a chunk of the env?
        case App("app", [App("lam", [x, body], lift), arg], lift2):
            env1 = clobber(x.thin, env, arg)
            return interp(body, env1)
        case App("app", [f, arg], lift):
            thin(lift, env)
            f1 = interp(f, env)

d is the depth
i is the index. I could instead index via d-i to use levels

I could have a smart constructor convention to just say differing contexts are left or right aligned and fill with False as necessary to reconcile

Yeah, maybe lam

If I construct my terms using db combinators, I end up with basically the same thing.

(named, nameless) hmm.
If I emulate locally nameless combinators, I need to maintain a cut point?

I can't really decidec if what I'm doing is orthogonal to de bruijn or it's just easy to translate from all of them.


In [52]:
def Var(d, i):
    assert 0 <= i < d
    t = [False]*d
    t[i] = True
    return App("var", [], t)

def VarLevel(d, l):
    return Var(d, d-l-1)

Var(2,0)

def LamDB(body):
    t = App("lam", [body])
    print(t)
    t.lift = t.lift[1:] # t.lift.pop()
    print(t)
    return t

LamDB(LamDB(Var(2,0)))

LTerm(lam, [LTerm(var, [], [True])], [True, False])
LTerm(lam, [LTerm(var, [], [True])], [False])
LTerm(lam, [LTerm(lam, [LTerm(var, [], [True])], [])], [False])
LTerm(lam, [LTerm(lam, [LTerm(var, [], [True])], [])], [])


LTerm(lam, [LTerm(lam, [LTerm(var, [], [True])], [])], [])

Semantics: Lambdas as dictionaries

x,y,z |- int ---> python frozenset families interp

sure I could implement the clobber version. the key in the dict gets clobbered and now has no effect. All keys are collected/grouped.



{}

dict[env, dict[x, ]]


How does one reorder variables in a lambda?
```
x,y,z |- t
-----
y,z |- l x, t


foo x y z = lam x.   no. Well we could clobber
foo x y z = (lam w, )(x)


x,y |- x*y = y*x


```





# Thin

In [ ]:


type Id = tuple[Thin, int]
@dataclass
class ThinUF:
    parents : list[Id] = field(default_factory=list)
    def makeset(self, scope : int) -> Id:
        i = len(self.parents)
        id = ([True]*scope, i)
        self.parents.append(id)
        return id
    def find(self, x : Id) -> Id:
        thin, xid = x
        while True:
            (thiny, yid) = self.parents[xid]
            if xid == yid:
                assert all(thiny)
                return (thin, xid)
            thin = comp(thiny, thin)
            xid = yid
    def union(self, x : Id, y : Id) -> Id | None:
        thinx, xid = self.find(x)
        thiny, yid = self.find(y)
        if xid != yid or thinx == thiny: 
            self.parents[xid] = yid
            return (thinx, xid)
        elif xid != yid or thinx != thiny:
            thinz = lcm(thinx,thiny)
            (_, z) = self.makeset(cod(thinz))
            self.parents[xid] = (div(thinz,thinx), z)
            self.parents[yid] = (div(thinz,thiny), z)
            return (thinz, z)
        else:
            # hmm
            return None
type Thin = tuple[bool, ...]
type Id = tuple[Thin, int]
def ctx(id : Id):
    return dom(id[0])
def widen(thin, x : Id) -> Id: # weaken
    return (comp(thin,x[0]), x[1])
# Based on https://github.com/mwillsey/microegg
class Pattern: ...
@dataclass(frozen=True)
class Var(Pattern):
    name: str
@dataclass(frozen=True)
class PApp(Pattern):
    f: str
    args: tuple[Pattern, ...]
type Node = tuple[str, tuple[Id, ...]]
type Subst = dict[str, Id]

# egraph

In [ ]:
@dataclass(frozen=True)
class Node:
    f : str
    args : tuple[Id,...]

@dataclass
class EGraph:
    uf: ThinUF = field(default_factory=ThinUF)
    memo: dict[Node, Id] = field(default_factory=dict)


    def add_node(self, obj: Node, n=0) -> Id:
        id = self.memo.get(obj)
        if id is not None:
            return self.find(id)
        else:
            id = self.uf.makeset(n)
            self.memo[obj] = id
            return id

    def app(self, f: str, *args: Id) -> Id:
        assert all(ctx(arg) == ctx(args[0]) for arg in args)
        # todo lift pulling
        return self.add_node(Node(f, args))

    def find(self, id: Id) -> Id:
        return self.uf.find(id)
    def union(self, id1: Id, id2: Id):
        return self.uf.union(id1, id2)        
    def nodes_in_class(self, id: Id) -> list[object]:
        id = self.find(id)
        return [obj for obj, obj_id in self.memo.items() if self.find(obj_id) == id]
    def is_eq(self, a: Id, b: Id) -> bool:
        return self.find(a) == self.find(b)
    def canonize_node(self, node: Node) -> Node:
        return Node(node.f, tuple(self.find(arg) for arg in node.args))

    def rebuild(self):
        copy_memo = self.memo.copy()
        while True:
            done = True
            for obj, id in copy_memo.items():
                id = self.find(id)
                new_node = self.canonize_node(obj)
                new_id = self._add(new_node)
                if new_id != id:
                    self.union(id, new_id)
                    done = False
            if done:
                return

    def ematch(self, pattern: Pattern, id: Id) -> list[Subst]:
        return self.ematch_rec(pattern, id, {})

    def ematch_rec(self, pattern: Pattern, id: Id, subst: Subst) -> list[Subst]:
        id = self.find(id)
        match pattern:
            case Var(name):
                if name in subst:
                    if self.is_eq(subst[name], id):
                        return [subst]
                    else:
                        return []
                else:
                    return [{**subst, name: id}]
            case PApp(f, args):
                results = []
                for obj in self.nodes_in_class(id):
                    match obj:
                        case (f0, arg_ids) if f0 == f and len(arg_ids) == len(args):
                            todo = [subst]
                            for arg_pattern, arg_id in zip(args, arg_ids):
                                todo = [
                                    subst1
                                    for subst0 in todo
                                    for subst1 in self.ematch_rec(
                                        arg_pattern, arg_id, subst0
                                    )
                                ]
                            results.extend(todo)
                        case _:
                            raise ValueError(f"Unexpected object in e-graph: {obj}")
                return results


E = EGraph()
a,b,c = E.app("a"), E.app("b"), E.app("c")
E.union(a,b)
E.union(b,c)
E.is_eq(a,c)
E

TypeError: cannot unpack non-iterable int object